In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.dpi": 130,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})
print("✅ Bibliotecas carregadas!")

In [ ]:
import os
ARQUIVO = [f for f in uploaded.keys() if f.endswith('.xlsx')][0]
print(f"📂 Arquivo carregado: {ARQUIVO}")

df_raw = pd.read_excel(ARQUIVO)


print("📐 Dimensões:", df_raw.shape)
print("🗺️  Estados:", df_raw["UF do Produto"].unique())
print("🗓️  Meses:", df_raw["Mês"].unique())
display(df_raw.head())

In [ ]:
anos = list(range(2010, 2027))
registros = []

for _, row in df_raw.iterrows():
    for ano in anos:
        col_fob = f"{ano} - Valor US$ FOB"
        col_kg  = f"{ano} - Quilograma Líquido"
        # CORREÇÃO 1: verifica AMBAS as colunas antes de acessar
        if col_fob in df_raw.columns and col_kg in df_raw.columns:
            registros.append({
                "mes_str"  : row["Mês"],
                "ano"      : ano,
                "uf"       : row["UF do Produto"],
                "valor_fob": row[col_fob],
                "kg_liq"   : row[col_kg],
            })


df = pd.DataFrame(registros)

mapa_mes = {
    "01. Janeiro":1, "02. Fevereiro":2, "03. Março":3,
    "04. Abril":4,   "05. Maio":5,      "06. Junho":6,
    "07. Julho":7,   "08. Agosto":8,    "09. Setembro":9,
    "10. Outubro":10,"11. Novembro":11, "12. Dezembro":12,
}
df["mes"] = df["mes_str"].map(mapa_mes)


df["data"] = pd.to_datetime({
    "year" : df["ano"],
    "month": df["mes"],
    "day"  : pd.Series(1, index=df.index),
})
df_completo = df[df["ano"] < 2026].copy()

print("✅ Dados transformados!")
print(f"   Total de linhas: {len(df)}")
display(df.head())

In [ ]:
serie_anual = (
    df_completo.groupby(["ano","uf"])[["valor_fob","kg_liq"]]
    .sum().reset_index()
)

df_sim = serie_anual.copy()
df_sim["projecao_10"] = df_sim["valor_fob"] * 1.10
display(df_sim.head())
serie_mensal = (
    df.groupby("data")[["valor_fob","kg_liq"]]
    .sum().reset_index().sort_values("data")
)
serie_estado = (
    df.groupby(["data","uf"])[["valor_fob","kg_liq"]]
    .sum().reset_index().sort_values("data")
)
sazonalidade = (
    df_completo.groupby(["mes","uf"])["valor_fob"]
    .mean().reset_index()
)
sazonalidade_total = (
    df_completo.groupby("mes")["valor_fob"]
    .mean().reset_index()
)
print("✅ Séries prontas!")

In [ ]:
# — Crescimento anual
cores = {"Bahia": "#2196F3", "Pernambuco": "#FF5722"}
nomes_mes = ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Crescimento Anual — Exportação de Manga\nBahia + Pernambuco (2010–2025)", fontsize=15, fontweight="bold")

ax = axes[0]
for uf, grupo in serie_anual.groupby("uf"):
    offset = 0.2 if uf == "Pernambuco" else -0.2
    ax.bar(grupo["ano"] + offset, grupo["valor_fob"]/1e6, width=0.4, label=uf, color=cores[uf], alpha=0.85)
ax.set_title("Valor Exportado (US$ FOB)")
ax.set_ylabel("Milhões de US$")
ax.legend()
# CORREÇÃO 3: range até 2026 para incluir o tick de 2024/2025
ax.set_xticks(range(2010, 2026, 2))
ax.set_xticklabels(range(2010, 2026, 2))

ax2 = axes[1]
for uf, grupo in serie_anual.groupby("uf"):
    grupo = grupo.sort_values("ano")
    base  = grupo[grupo["ano"] == 2010]["valor_fob"].values[0]

    cresc = (grupo["valor_fob"].values / base - 1) * 100
    ax2.plot(grupo["ano"].values, cresc, marker="o", label=uf, color=cores[uf], linewidth=2.5)
ax2.axhline(0, color="gray", linestyle="--")
ax2.set_title("Crescimento Acumulado (base 2010)")
ax2.set_ylabel("Variação %")
ax2.legend()
ax2.set_xticks(range(2010, 2026, 2))
ax2.set_xticklabels(range(2010, 2026, 2))

plt.tight_layout()
plt.show()

In [ ]:
#  Sazonalidade
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Sazonalidade — Exportação de Manga\nMédia Mensal (2010–2025)", fontsize=15, fontweight="bold")

ax = axes[0]
for uf, grupo in sazonalidade.groupby("uf"):
    ax.plot(grupo["mes"], grupo["valor_fob"]/1e6, marker="o", label=uf, color=cores[uf], linewidth=2.5)
ax.set_xticks(range(1, 13))
ax.set_xticklabels(nomes_mes)
ax.set_title("Média por Estado")
ax.set_ylabel("Milhões de US$")
ax.legend()

ax2 = axes[1]

valores_norm = sazonalidade_total["valor_fob"].values / sazonalidade_total["valor_fob"].max()
colors_bar = plt.cm.YlOrRd(valores_norm)
bars = ax2.bar(sazonalidade_total["mes"], sazonalidade_total["valor_fob"]/1e6, color=colors_bar, edgecolor="white")
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(nomes_mes)
ax2.set_title("Total BA + PE — Intensidade por Cor")
ax2.set_ylabel("Milhões de US$")
for bar, val in zip(bars, sazonalidade_total["valor_fob"]/1e6):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.1, f"{val:.1f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()


In [ ]:
# Heatmap
pivot_raw = df_completo.groupby(["ano","mes"])["valor_fob"].sum().unstack(level="mes")
# CORREÇÃO 6: reindexar para garantir exatamente 12 colunas (meses 1–12)
pivot = pivot_raw.reindex(columns=range(1, 13))
pivot.columns = nomes_mes

fig, ax = plt.subplots(figsize=(14, 8))
sns.heatmap(pivot/1e6, annot=True, fmt=".0f", cmap="YlOrRd",
            linewidths=0.5, ax=ax, cbar_kws={"label": "US$ Milhões"})
ax.set_title("Heatmap: Exportação por Mês × Ano\nBahia + Pernambuco (US$ Milhões)", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Série temporal completa
fig, ax = plt.subplots(figsize=(16, 5))

for uf in ["Bahia", "Pernambuco"]:
    dados = serie_estado[serie_estado["uf"] == uf]
    ax.plot(dados["data"], dados["valor_fob"] / 1e6,
            label=uf, color=cores[uf], linewidth=1.8, alpha=0.85)


serie_mensal_plot = serie_mensal.copy()
serie_mensal_plot["mm12"] = serie_mensal_plot["valor_fob"].rolling(12).mean()
ax.plot(serie_mensal_plot["data"], serie_mensal_plot["mm12"] / 1e6,
        color="black", linewidth=2.5, linestyle="--", label="Média Móvel 12m (Total)")

ax.set_title(
    "Série Temporal — Exportação de Manga (2010–2026)",
    fontsize=14, fontweight="bold"
)
ax.set_ylabel("US$ Milhões (FOB)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"US$ {x:.0f}M"))
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

for uf in df_sim["uf"].unique():
    dados = df_sim[df_sim["uf"] == uf]

    plt.plot(dados["ano"], dados["valor_fob"]/1e6, label=f"{uf} Real")
    plt.plot(dados["ano"], dados["projecao_10"]/1e6, linestyle="--", label=f"{uf} +10%")

plt.title("Real vs Projeção (+10%)")
plt.ylabel("US$ Milhões")
plt.legend()
plt.show()

In [ ]:
#  Insights finais
total_anual = serie_anual.groupby("ano")["valor_fob"].sum()
top_mes     = sazonalidade_total.sort_values("valor_fob", ascending=False).iloc[0]
bot_mes     = sazonalidade_total.sort_values("valor_fob").iloc[0]


val_2010 = total_anual.get(2010)
val_2025 = total_anual.get(2025)

melhor_ano  = total_anual.idxmax()

print("=" * 50)
print("📊 PRINCIPAIS INSIGHTS")
print("=" * 50)
print(f"📅 Mês mais forte : {nomes_mes[int(top_mes['mes'])-1]} (média US$ {top_mes['valor_fob']/1e6:.1f}M)")
print(f"📉 Mês mais fraco : {nomes_mes[int(bot_mes['mes'])-1]} (média US$ {bot_mes['valor_fob']/1e6:.1f}M)")

if val_2010 is not None and val_2025 is not None:
    cresc = (val_2025 / val_2010 - 1) * 100
    print(f"📈 Crescimento 2010→2025 : +{cresc:.0f}%")
    print(f"   2010 : US$ {val_2010/1e6:.1f}M")
    print(f"   2025 : US$ {val_2025/1e6:.1f}M")
else:
    print("⚠️  Dados de 2010 ou 2025 não disponíveis para calcular crescimento.")

print(f"🏆 Melhor ano : {melhor_ano} (US$ {total_anual[melhor_ano]/1e6:.1f}M)")


In [ ]:
df.to_csv("dados_tratados.csv", index=False)
df_sim.to_csv("simulacao.csv", index=False)

In [ ]:
df_sim.to_csv("simulacao.csv", index=False)

In [ ]:
from google.colab import files

files.download("dados_tratados.csv")
files.download("simulacao.csv")